# Formation Energy and Band Structure of the Pyridinic N₃-Vacancy Defect in Graphene

> **Yoshitaka Fujimoto and Susumu Saito**, "Formation, stabilities, and electronic properties of
> nitrogen defects in graphene", Physical Review B 84, 245446 (2011).
> [DOI:10.1103/PhysRevB.84.245446](https://doi.org/10.1103/PhysRevB.84.245446)

Computes the formation energy and the band structure of the trimerized pyridine-type C₂₈N₃ defect
created in the [structure notebook](defect_point_substitution_graphene.ipynb), both off the same
cell, and compares the formation energy with the manuscript's Table I entry, 2.51 eV.

<h2 style="color:green">Usage</h2>

1. Create the materials in the [structure notebook](defect_point_substitution_graphene.ipynb), which saves `graphene 4x4` and `graphene 4x4 N3V pyridinic (C28N3)` to the `uploads` folder.
1. Set the material names and parameters in cells 1.2-1.4 (or use the defaults); `RELAX = True` uses the relaxed structure, relaxing once and saving it as `<name> relaxed` for reuse by this and other notebooks.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the jobs to complete.
1. Scroll down to view the results.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the material names, model and compute parameters.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the materials by name from the `uploads` folder, resolve the nitrogen reference, and save them to the platform.
1. Configure the shared DFT model and k-grid: one model and a per-material k-grid for every job below.
1. Configure compute: get the list of clusters and create a compute configuration.
1. Relax the defective cell if `RELAX`, reusing a saved relaxed structure when one already exists.
1. Create the missing Total Energy jobs for the defective, pristine and nitrogen cells and wait for them.
1. Configure the Band Structure workflow and run it on the defective cell.
1. Retrieve the band structure, assemble the formation energy and compare it with the manuscript.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples|api_examples")

### 1.2. Material names

In [ ]:
# Names saved by defect_point_substitution_graphene.ipynb.
PRISTINE_NAME = "graphene 4x4"
DEFECTIVE_NAME = "graphene 4x4 N3V pyridinic (C28N3)"
# Standata's solid nitrogen -- the platform carries no free N2 molecule.
NITROGEN_NAME = "N2, Nitrogen, FCC (P2_13) 3D (Bulk), mp-154"

### 1.3. Parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

TOTAL_ENERGY_SEARCH_TERM = "total_energy.json"
RELAX_WORKFLOW_SEARCH_TERM = "fixed_cell_relaxation.json"
BAND_STRUCTURE_WORKFLOW_SEARCH_TERM = "band_structure.json"
MY_WORKFLOW_NAME = "Band Structure"
APPLICATION_NAME = "espresso"
TOTAL_ENERGY_SOURCE = "my_account"

# False: use the structures as given, fast. True: use the relaxed defective structure, running the
# relaxation once if it does not exist yet.
RELAX = False

CLUSTER_NAME = "001"  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.OR
PPN = 40
TIME_LIMIT = "12:00:00"  # covers the optional relaxation (~1 h)

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.4. DFT model parameters

In [ ]:
MODEL_SUBTYPE = "lda"
FUNCTIONAL = "pz"
PSEUDOPOTENTIAL_TYPE = "us"  # GBRV ultrasoft, the only LDA family the platform publishes for C and N
ECUTWFC = 50   # Ry, Fujimoto & Saito Sec. II
ECUTRHO = 200  # Ry, GBRV's recommended charge-density cutoff

KPOINT_DENSITY = 7  # points per Å⁻¹; gives 6 x 6 x 1 on the 4x4 cell, Fujimoto & Saito Sec. II
MODEL_TAG = f"{FUNCTIONAL}-{PSEUDOPOTENTIAL_TYPE} {ECUTWFC}-{ECUTRHO}Ry k{KPOINT_DENSITY}"

SCF_UNIT = "pw_scf"
BANDS_UNIT = "pw_bands"
RELAX_UNIT = "pw_relax"
# C28N3 carries 127 valence electrons, so its ground state is a doublet.
SPIN_SETTINGS = {
    DEFECTIVE_NAME: {"nspin": 2, "tot_magnetization": 1},
    PRISTINE_NAME: {"nspin": 1},
    NITROGEN_NAME: {"nspin": 1},
}
RELAXATION_SETTINGS = {"forc_conv_thr": 1.9e-3, "nstep": 100}  # 0.05 eV/Å, Fujimoto & Saito Sec. II
# Names the relaxation job; the relaxed structure itself is found by content hash.
RELAX_TAG = f"{MODEL_TAG} nspin2 f{RELAXATION_SETTINGS['forc_conv_thr']}"

KPATH_STEPS = 20
KPATH = [
    {"point": "K", "steps": KPATH_STEPS},
    {"point": "Γ", "steps": KPATH_STEPS},
    {"point": "M", "steps": KPATH_STEPS},
    {"point": "K", "steps": 1},
]

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Load the materials
### 3.1. Load from the uploads folder or the platform, and print provenance

In [ ]:
from collections import Counter
from mat3ra.notebooks_utils.core.entity.material.api import load_material

materials_by_name = {name: load_material(client, FOLDER, name, ACCOUNT_ID) for name in (PRISTINE_NAME, DEFECTIVE_NAME)}
pristine, defective = materials_by_name[PRISTINE_NAME], materials_by_name[DEFECTIVE_NAME]
for name, material in materials_by_name.items():
    composition = "".join(f"{e}{n}" for e, n in sorted(Counter(material.basis.elements.values).items()))
    print(f"{name}: {composition}, {material.basis.number_of_atoms} atoms, "
          f"cell {material.lattice.a:.2f} x {material.lattice.b:.2f} x {material.lattice.c:.2f} Å")

### 3.2. Resolve the nitrogen reference material

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials

nitrogen = Material.create(Materials.get_by_name_first_match(NITROGEN_NAME))
print(f"{nitrogen.name}: {nitrogen.basis.number_of_atoms} atoms, cell {nitrogen.lattice.a:.2f} Å")

### 3.3. Save the materials to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_pristine = get_or_create_material(client, pristine, ACCOUNT_ID)
saved_defective = get_or_create_material(client, defective, ACCOUNT_ID)
saved_nitrogen = get_or_create_material(client, nitrogen, ACCOUNT_ID)

## 4. Configure the shared DFT model and k-grid
### 4.1. DFT model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
print(f"Using application: {app.name}, model: {MODEL_TAG}")

### 4.2. k-grid per material

In [ ]:
from mat3ra.notebooks_utils.workflow import kgrid_from_density

material_objects = {PRISTINE_NAME: pristine, DEFECTIVE_NAME: defective, NITROGEN_NAME: nitrogen}
kgrid = {
    PRISTINE_NAME: kgrid_from_density(pristine, KPOINT_DENSITY, periodic_dims=(0, 1)),
    DEFECTIVE_NAME: kgrid_from_density(defective, KPOINT_DENSITY, periodic_dims=(0, 1)),
    NITROGEN_NAME: kgrid_from_density(nitrogen, KPOINT_DENSITY),
}
for name, grid in kgrid.items():
    print(f"{name}: k-grid {grid}")

## 5. Create the compute configuration
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. Available: {[c['hostname'] for c in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")

## 6. Relax the defective cell (optional)

Runs only if `RELAX`: finds any relaxed version of this structure already on the account,
regardless of who relaxed it or with what -- before spending any compute on a new one. The band
structure and the total energy are then both taken off that one geometry.

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.workflow import apply_planewave_cutoffs, apply_scf_kgrid, patch_workflow_qe_input
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material
from mat3ra.notebooks_utils.core.entity.material.api import find_relaxed_material, get_final_structure_for_job
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

relaxed_defective = None
if RELAX:
    relax_workflow_name = f"Fixed-cell Relaxation {DEFECTIVE_NAME} {RELAX_TAG}"
    relaxed_defective = find_relaxed_material(client, defective, ACCOUNT_ID)
    if relaxed_defective is not None:
        print(f"♻️  Relaxed defective material: {relaxed_defective.name} ({relaxed_defective.id})")
    else:
        relax_workflow = Workflow.create(
            WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(RELAX_WORKFLOW_SEARCH_TERM)
        )
        relax_workflow.name = relax_workflow_name
        relax_workflow.subworkflows[0].model = model
        apply_planewave_cutoffs(relax_workflow, ECUTWFC, ECUTRHO, unit_name=RELAX_UNIT)
        apply_scf_kgrid(relax_workflow, kgrid[DEFECTIVE_NAME], material=defective, unit_name=RELAX_UNIT)
        patch_workflow_qe_input(relax_workflow, {"system": SPIN_SETTINGS[DEFECTIVE_NAME]}, [RELAX_UNIT])
        patch_workflow_qe_input(relax_workflow, {"control": RELAXATION_SETTINGS}, [RELAX_UNIT])

        relax_job = find_job_for_material(
            client, saved_defective["_id"], relax_workflow_name, ACCOUNT_ID,
            statuses=("submitted", "queued", "active", "finished"),
        )
        if relax_job is None:
            relax_job = create_job(
                api_client=client, materials=[saved_defective], workflow=relax_workflow,
                project_id=project_id, owner_id=ACCOUNT_ID, compute=compute.to_dict(),
                prefix=f"{relax_workflow_name} {timestamp}",
            )
            submit_jobs(client.jobs, [relax_job["_id"]])
        await wait_for_jobs_to_finish_async(client.jobs, [relax_job["_id"]], poll_interval=POLL_INTERVAL)

        relaxed_material = get_final_structure_for_job(client, relax_job["_id"])
        client.materials.update(relaxed_material.id, {"name": f"{DEFECTIVE_NAME} relaxed"})
        relaxed_defective = Material.create(client.materials.get(relaxed_material.id))
        print(f"✅ Relaxed defective material: {relaxed_defective.name} ({relaxed_defective.id})")

        total_force = get_properties_for_job(client, relax_job["_id"], "total_force")[0]
        print(f"Residual force after relaxation (norm over all atoms): "
              f"{total_force['value']:.4f} {total_force['units']}")

defective_for_jobs = relaxed_defective if RELAX else defective
defective_for_jobs.lattice.type = "HEX"  # the symbolic K-Γ-M-K path needs a named lattice type
material_objects[DEFECTIVE_NAME] = defective_for_jobs
saved_defective_for_jobs = get_or_create_material(client, defective_for_jobs, ACCOUNT_ID)

## 7. Total Energy jobs for the defective, pristine and nitrogen cells

In [ ]:
from mat3ra.notebooks_utils.core.entity.property.api import find_total_energy_for_material

total_energy_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    TOTAL_ENERGY_SEARCH_TERM
)
saved_materials = {
    DEFECTIVE_NAME: saved_defective_for_jobs,
    PRISTINE_NAME: saved_pristine,
    NITROGEN_NAME: saved_nitrogen,
}
total_energies = {}
total_energy_job_ids = {}
new_job_ids = []
for name, saved_material in saved_materials.items():
    existing = find_total_energy_for_material(client, saved_material["_id"], source=TOTAL_ENERGY_SOURCE)
    if existing is not None:
        total_energies[name] = existing["data"]["value"]
        print(f"♻️  {name}: reusing total energy {total_energies[name]:.4f} eV")
        continue
    workflow = Workflow.create(total_energy_workflow_config)
    workflow.name = f"Total Energy {name} {MODEL_TAG}"
    workflow.subworkflows[0].model = model
    apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=SCF_UNIT)
    apply_scf_kgrid(workflow, kgrid[name], material=material_objects[name])
    patch_workflow_qe_input(workflow, {"system": SPIN_SETTINGS[name]}, [SCF_UNIT])
    job = create_job(
        api_client=client, materials=[saved_material], workflow=workflow, project_id=project_id,
        owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}",
    )
    total_energy_job_ids[name] = job["_id"]
    new_job_ids.append(job["_id"])
    print(f"✅ {name}: created Total Energy job {job['_id']}")

In [ ]:
if new_job_ids:
    submit_jobs(client.jobs, new_job_ids)
    print(f"✅ Submitted {len(new_job_ids)} Total Energy job(s).")
    await wait_for_jobs_to_finish_async(client.jobs, new_job_ids, poll_interval=POLL_INTERVAL)

for name, job_id in total_energy_job_ids.items():
    total_energies[name] = get_properties_for_job(client, job_id, "total_energy")[0]["value"]
    print(f"{name}: {total_energies[name]:.4f} eV")

## 8. Band structure of the defective cell
### 8.1. Configure the workflow

In [ ]:
from mat3ra.wode.context.providers import PointsPathDataProvider
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

band_structure_workflow = Workflow.create(
    WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(BAND_STRUCTURE_WORKFLOW_SEARCH_TERM)
)
band_structure_workflow.name = f"{MY_WORKFLOW_NAME} {DEFECTIVE_NAME} {MODEL_TAG}"
band_structure_workflow.subworkflows[0].model = model
apply_planewave_cutoffs(band_structure_workflow, ECUTWFC, ECUTRHO, unit_name=SCF_UNIT)
apply_planewave_cutoffs(band_structure_workflow, ECUTWFC, ECUTRHO, unit_name=BANDS_UNIT)
apply_scf_kgrid(band_structure_workflow, kgrid[DEFECTIVE_NAME], material=defective_for_jobs)
patch_workflow_qe_input(band_structure_workflow, {"system": SPIN_SETTINGS[DEFECTIVE_NAME]}, [SCF_UNIT, BANDS_UNIT])

bands_subworkflow = band_structure_workflow.subworkflows[0]
bands_unit = bands_subworkflow.get_unit_by_name(name=BANDS_UNIT)
bands_unit.add_context(PointsPathDataProvider(path=KPATH, isEdited=True).get_context_item_data())
bands_subworkflow.set_unit(bands_unit)

visualize_workflow(band_structure_workflow)

### 8.2. Create the job

In [ ]:
band_structure_job = create_job(
    api_client=client, materials=[saved_defective_for_jobs], workflow=band_structure_workflow,
    project_id=project_id, owner_id=ACCOUNT_ID, compute=compute.to_dict(),
    prefix=f"{band_structure_workflow.name} {timestamp}",
)
band_structure_job_id = band_structure_job["_id"]
print(f"✅ Band Structure job created: {band_structure_job_id}")

### 8.3. Submit and monitor the job

In [ ]:
client.jobs.submit(band_structure_job_id)
print(f"✅ Job {band_structure_job_id} submitted successfully!")
await wait_for_jobs_to_finish_async(client.jobs, [band_structure_job_id], poll_interval=POLL_INTERVAL)

## 9. Retrieve the results
### 9.1. Band structure

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

band_structure_data = get_properties_for_job(client, band_structure_job_id, property_name="band_structure")
visualize_properties(band_structure_data, title="Band Structure",
                     extra_config={"material": defective_for_jobs.to_dict()})

### 9.2. Formation energy

In [ ]:
defective_composition = Counter(defective_for_jobs.basis.elements.values)
mu_carbon = total_energies[PRISTINE_NAME] / pristine.basis.number_of_atoms
mu_nitrogen = total_energies[NITROGEN_NAME] / nitrogen.basis.number_of_atoms
e_formation = (
    total_energies[DEFECTIVE_NAME]
    - defective_composition["C"] * mu_carbon
    - defective_composition["N"] * mu_nitrogen
)

print(f"μ_C = {mu_carbon:.4f} eV/atom ({pristine.basis.number_of_atoms} atoms)")
print(f"μ_N = {mu_nitrogen:.4f} eV/atom ({nitrogen.basis.number_of_atoms} atoms)")
print(f"E_f = {total_energies[DEFECTIVE_NAME]:.4f} eV - {defective_composition['C']} μ_C "
      f"- {defective_composition['N']} μ_N = {e_formation:.3f} eV")

### 9.3. Compare with Fujimoto & Saito (2011)

In [ ]:
FUJIMOTO_FORMATION_ENERGY = 2.51  # eV, Table I, trimerized pyridine-type C28N3
TOLERANCE_FRACTION = 0.15

deviation = e_formation - FUJIMOTO_FORMATION_ENERGY
verdict = "yes" if abs(deviation) <= TOLERANCE_FRACTION * FUJIMOTO_FORMATION_ENERGY else "no"
regime = "relaxed" if RELAX else "unrelaxed"
print(f"E_f (this notebook):                  {e_formation:.3f} eV")
print(f"E_f (Fujimoto & Saito 2011, Table I): {FUJIMOTO_FORMATION_ENERGY:.3f} eV")
print(f"Deviation: {deviation:+.3f} eV ({100 * deviation / FUJIMOTO_FORMATION_ENERGY:+.1f} %, "
      f"tolerance {100 * TOLERANCE_FRACTION:.0f} %)")
print("Offsets: μ_N comes from solid N2 (mp-154) rather than the free molecule the manuscript "
      "uses, and GBRV ultrasoft LDA stands in for its Troullier-Martins norm-conserving set.")
print(f"Reproduces Fujimoto & Saito (2011): {verdict} ({regime})")

## References

[1] Y. Fujimoto and S. Saito, "Formation, stabilities, and electronic properties of nitrogen defects in graphene", Phys. Rev. B 84, 245446 (2011). https://doi.org/10.1103/PhysRevB.84.245446

[2] K. F. Garrity, J. W. Bennett, K. M. Rabe and D. Vanderbilt, "Pseudopotentials for high-throughput DFT calculations", Comput. Mater. Sci. 81, 446 (2014). https://doi.org/10.1016/j.commatsci.2013.08.053